# XGBoost + Autoencoder Hybrid — Credit Card Only (Ensemble Anchor)

**Strategy:** Train a vanilla AE on normal-only data, use reconstruction error as additional
features for XGBoost. **Target:** F1 ≈ 0.94. **Tag:** `xgb_ae_hybrid`


## Cell 0 — Setup

In [1]:
# Cell 0 — Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'xgboost'])

import os, json, warnings
import numpy as np
import pandas as pd
import joblib

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve, classification_report,
)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

RUNID = 'ensemble_run_001'
DRIVEROOT = '/content/drive/MyDrive/tsad_ensemble_runs'
NOTEBOOKTAG = 'xgb_ae_hybrid'

RUNDIR = os.path.join(DRIVEROOT, RUNID, NOTEBOOKTAG)
ARTIFACTDIR = os.path.join(RUNDIR, 'artifacts')
PREDICTIONSDIR = os.path.join(RUNDIR, 'predictions')
CACHEDIR = os.path.join(DRIVEROOT, '_cache')

for d in [ARTIFACTDIR, PREDICTIONSDIR, CACHEDIR]:
    os.makedirs(d, exist_ok=True)

print('RUNDIR       :', RUNDIR)
print('ARTIFACTDIR  :', ARTIFACTDIR)
print('PREDICTIONSDIR:', PREDICTIONSDIR)
print('XGBoost version:', xgb.__version__)


Mounted at /content/drive
RUNDIR       : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/xgb_ae_hybrid
ARTIFACTDIR  : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/xgb_ae_hybrid/artifacts
PREDICTIONSDIR: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/xgb_ae_hybrid/predictions
XGBoost version: 3.2.0


## Cell 1 — Load Data & Feature Engineering

In [2]:
# Cell 1 — Load Data & Feature Engineering

MYDRIVE = '/content/drive/MyDrive'
CREDITCARDPATH = os.path.join(MYDRIVE, 'creditcard.csv')
assert os.path.exists(CREDITCARDPATH), f'creditcard.csv not found at {CREDITCARDPATH}'

print('Loading Credit Card data...')
df = pd.read_csv(CREDITCARDPATH).dropna().reset_index(drop=True)
y_all = df['Class'].astype(int).values

X_df = df.copy()
X_df['hoursin'] = np.sin(2 * np.pi * X_df['Time'] / 86400.0)
X_df['hourcos'] = np.cos(2 * np.pi * X_df['Time'] / 86400.0)
X_df['Amountlog'] = np.log1p(X_df['Amount'])
X_df['AmountSq'] = X_df['Amount'] ** 2
for v in ['V1', 'V3', 'V4', 'V7', 'V10', 'V12', 'V14', 'V17']:
    X_df[f'{v}_abs'] = X_df[v].abs()

base_feature_cols = (
    [f'V{i}' for i in range(1, 29)]
    + ['Amountlog', 'AmountSq', 'hoursin', 'hourcos']
    + [f'{v}_abs' for v in ['V1', 'V3', 'V4', 'V7', 'V10', 'V12', 'V14', 'V17']]
)

X_base = X_df[base_feature_cols].values.astype(np.float64)

print(f'Loaded {len(df):,} rows, {len(base_feature_cols)} base features')
print(f'Fraud count: {y_all.sum()} ({100*y_all.mean():.4f}%)')


Loading Credit Card data...
Loaded 284,807 rows, 40 base features
Fraud count: 492 (0.1727%)


## Cell 2 — Train/Val/Test Split (60/20/20 stratified)

In [3]:
# Cell 2 — Train/Val/Test Split

idx = np.arange(len(y_all), dtype=np.int64)

idx_tune, idx_test = train_test_split(
    idx, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE
)
idx_train, idx_val = train_test_split(
    idx_tune, test_size=0.25, stratify=y_all[idx_tune], random_state=RANDOM_STATE
)

X_train_all = X_base[idx_train]
y_train = y_all[idx_train]
X_val_all = X_base[idx_val]
y_val = y_all[idx_val]
X_test_all = X_base[idx_test]
y_test = y_all[idx_test]

X_train_normal = X_train_all[y_train == 0]

print(f'Train     : {len(X_train_all):,} rows ({y_train.sum()} frauds)')
print(f'  Normal  : {len(X_train_normal):,} rows (for AE training)')
print(f'Validation: {len(X_val_all):,} rows ({y_val.sum()} frauds)')
print(f'Test      : {len(X_test_all):,} rows ({y_test.sum()} frauds)')

assert len(set(idx_train) & set(idx_val)) == 0
assert len(set(idx_train) & set(idx_test)) == 0
assert len(set(idx_val) & set(idx_test)) == 0
print('No overlap between splits. OK.')


Train     : 170,883 rows (295 frauds)
  Normal  : 170,588 rows (for AE training)
Validation: 56,962 rows (99 frauds)
Test      : 56,962 rows (98 frauds)
No overlap between splits. OK.


## Cell 3 — Train Vanilla Autoencoder on Normal Data

**v2 fix:** Clip scaled features to [-5, 5] before AE to prevent extreme values
(especially AmountSq) from dominating reconstruction error and causing
normal MSE > fraud MSE.

In [4]:
# Cell 3 — Train Vanilla Autoencoder

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

tf.random.set_seed(RANDOM_STATE)

# Scale + clip for AE stability
ae_scaler = RobustScaler()
X_train_normal_sc = np.clip(ae_scaler.fit_transform(X_train_normal), -5, 5)
X_train_all_sc = np.clip(ae_scaler.transform(X_train_all), -5, 5)
X_val_all_sc = np.clip(ae_scaler.transform(X_val_all), -5, 5)
X_test_all_sc = np.clip(ae_scaler.transform(X_test_all), -5, 5)

input_dim = X_train_normal_sc.shape[1]
print(f'AE input dimension: {input_dim}')

def build_autoencoder(input_dim):
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(32, activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(16, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(8, activation='relu')(x)
    x = layers.Dense(16, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    out = layers.Dense(input_dim, activation='linear')(x)
    return keras.Model(inp, out, name='vanilla_ae')

ae_model = build_autoencoder(input_dim)
ae_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse')
ae_model.summary()

print('\nTraining AE on normal data only...')
ae_history = ae_model.fit(
    X_train_normal_sc, X_train_normal_sc,
    epochs=100, batch_size=512, validation_split=0.1,
    callbacks=[
        callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6),
    ],
    verbose=1
)

print(f'AE training complete. Best val loss: {min(ae_history.history["val_loss"]):.6f}')


AE input dimension: 40


Model: "vanilla_ae"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 40)             │         1,320 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,368 (17.06 KB)

 Trainable params: 4,176 (16.31 KB)

 Non-trainable params: 192 (768.00 B)


Training AE on normal data only...
Epoch 1/100
300/300 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 0.8378 - val_loss: 0.6255 - learning_rate: 0.0010
Epoch 2/100
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.5871 - val_loss: 0.5113 - learning_rate: 0.0010
Epoch 3/100
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.5087 - val_loss: 0.4473 - learning_rate: 0.0010
Epoch 4/100
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4620 - val_loss: 0.4045 - learning_rate: 0.0010
Epoch 5/100
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.4320 - val_loss: 0.3789 - learning_rate: 0.0010
Epoch 6/100
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.4106 - val_loss: 0.3586 - learning_rate: 0.0010
Epoch 7/100
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.3942 - val_loss: 0.3423 - learning_rate: 0.0010
Epoch 8/100
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.3813 - val_loss: 0.3305 - learning_rate: 0.0010
Epoch 9/100
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.3721 - val_lo

## Cell 4 — Compute AE Reconstruction Error Features

In [5]:
# Cell 4 — Compute AE Reconstruction Error Features

def compute_ae_features(ae_model, X_scaled):
    X_recon = ae_model.predict(X_scaled, batch_size=4096, verbose=0)
    errors = X_scaled - X_recon
    mse = np.mean(errors ** 2, axis=1)
    mae = np.mean(np.abs(errors), axis=1)
    max_err = np.max(np.abs(errors), axis=1)
    return np.column_stack([mse, mae, max_err])

print('Computing AE reconstruction error features...')
ae_feat_train = compute_ae_features(ae_model, X_train_all_sc)
ae_feat_val = compute_ae_features(ae_model, X_val_all_sc)
ae_feat_test = compute_ae_features(ae_model, X_test_all_sc)

ae_feature_names = ['ae_mse', 'ae_mae', 'ae_max_err']
all_feature_cols = base_feature_cols + ae_feature_names

X_train_hybrid = np.hstack([X_train_all, ae_feat_train])
X_val_hybrid = np.hstack([X_val_all, ae_feat_val])
X_test_hybrid = np.hstack([X_test_all, ae_feat_test])

print(f'Hybrid feature count: {X_train_hybrid.shape[1]} ({len(base_feature_cols)} base + {len(ae_feature_names)} AE)')
print(f'\nAE MSE — Normal train mean: {ae_feat_train[y_train == 0, 0].mean():.6f}')
print(f'AE MSE — Fraud  train mean: {ae_feat_train[y_train == 1, 0].mean():.6f}')
print(f'AE MSE — Normal val   mean: {ae_feat_val[y_val == 0, 0].mean():.6f}')
print(f'AE MSE — Fraud  val   mean: {ae_feat_val[y_val == 1, 0].mean():.6f}')
ratio = ae_feat_val[y_val == 1, 0].mean() / (ae_feat_val[y_val == 0, 0].mean() + 1e-12)
print(f'Fraud/Normal MSE ratio (val): {ratio:.1f}x — should be >1')


Computing AE reconstruction error features...
Hybrid feature count: 43 (40 base + 3 AE)

AE MSE — Normal train mean: 0.236503
AE MSE — Fraud  train mean: 5.946571
AE MSE — Normal val   mean: 0.239071
AE MSE — Fraud  val   mean: 5.260414
Fraud/Normal MSE ratio (val): 22.0x — should be >1


## Cell 5 — Train XGBoost on Hybrid Features

**v2 fix:** `early_stopping_rounds=100` set as constructor param (works in xgboost ≥2.0).

In [6]:
# Cell 5 — Train XGBoost on Hybrid Features

pos = int(y_train.sum())
neg = int((y_train == 0).sum())
spw = neg / pos
print(f'scale_pos_weight: {spw:.2f} ({neg} neg / {pos} pos)')

xgb_model = xgb.XGBClassifier(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1.0,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=spw,
    tree_method='hist',
    eval_metric='logloss',
    early_stopping_rounds=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print('Training XGBoost on hybrid features...')
xgb_model.fit(
    X_train_hybrid, y_train,
    eval_set=[(X_val_hybrid, y_val)],
    verbose=200,
)

try:
    print(f'Best iteration: {xgb_model.best_iteration}')
    print(f'Best score: {xgb_model.best_score:.6f}')
except AttributeError:
    print(f'Trained all {xgb_model.n_estimators} rounds.')


scale_pos_weight: 578.26 (170588 neg / 295 pos)
Training XGBoost on hybrid features...
[0]	validation_0-logloss:0.67448
[200]	validation_0-logloss:0.02005
[400]	validation_0-logloss:0.00592
[600]	validation_0-logloss:0.00405
[800]	validation_0-logloss:0.00360
[1000]	validation_0-logloss:0.00356
[1054]	validation_0-logloss:0.00355
Best iteration: 954
Best score: 0.003550


## Cell 6 — Threshold Tuning

In [7]:
# Cell 6 — Threshold Tuning

val_probs = xgb_model.predict_proba(X_val_hybrid)[:, 1]

prec, rec, thr = precision_recall_curve(y_val, val_probs)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_idx = int(np.nanargmax(f1s))
best_threshold = float(thr[best_idx])
best_val_f1 = float(f1s[best_idx])

print(f'PR-curve best threshold: {best_threshold:.6f}')
print(f'Validation F1 at threshold: {best_val_f1:.4f}')

print('\nThreshold grid search:')
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    p = (val_probs >= t).astype(int)
    f = f1_score(y_val, p, zero_division=0)
    pr = precision_score(y_val, p, zero_division=0)
    re = recall_score(y_val, p, zero_division=0)
    print(f'  t={t:.2f}  P={pr:.4f}  R={re:.4f}  F1={f:.4f}')

print(f'\n>>> Using threshold = {best_threshold:.6f}')


PR-curve best threshold: 0.824807
Validation F1 at threshold: 0.8462

Threshold grid search:
  t=0.30  P=0.8316  R=0.7980  F1=0.8144
  t=0.40  P=0.8556  R=0.7778  F1=0.8148
  t=0.50  P=0.8652  R=0.7778  F1=0.8191
  t=0.60  P=0.8851  R=0.7778  F1=0.8280
  t=0.70  P=0.9059  R=0.7778  F1=0.8370
  t=0.80  P=0.9167  R=0.7778  F1=0.8415
  t=0.90  P=0.9268  R=0.7677  F1=0.8398
  t=0.95  P=0.9375  R=0.7576  F1=0.8380

>>> Using threshold = 0.824807


## Cell 7 — Test Evaluation

In [8]:
# Cell 7 — Test Evaluation

test_probs = xgb_model.predict_proba(X_test_hybrid)[:, 1]
test_preds = (test_probs >= best_threshold).astype(np.int8)

precision_v = float(precision_score(y_test, test_preds, zero_division=0))
recall_v = float(recall_score(y_test, test_preds, zero_division=0))
f1_v = float(f1_score(y_test, test_preds, zero_division=0))
rocauc_v = float(roc_auc_score(y_test, test_probs))
prauc_v = float(average_precision_score(y_test, test_probs))

print('=' * 60)
print('XGBOOST + AE HYBRID — TEST RESULTS')
print('=' * 60)
print(f'  Precision    : {precision_v:.4f}')
print(f'  Recall       : {recall_v:.4f}')
print(f'  F1           : {f1_v:.4f}')
print(f'  ROC-AUC      : {rocauc_v:.4f}')
print(f'  PR-AUC       : {prauc_v:.4f}')
print(f'  Threshold    : {best_threshold:.6f}')
print(f'  True frauds  : {int(y_test.sum())}')
print(f'  Pred frauds  : {int(test_preds.sum())}')
print()
print('Classification Report:')
print(classification_report(y_test, test_preds, target_names=['Normal', 'Fraud']))

importances = xgb_model.feature_importances_
fi_df = pd.DataFrame({'feature': all_feature_cols, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(15)
print('\nTop 15 Feature Importances:')
for _, row in fi_df.iterrows():
    print(f"  {row['feature']:20s}: {row['importance']:.4f}")


XGBOOST + AE HYBRID — TEST RESULTS
  Precision    : 0.9101
  Recall       : 0.8265
  F1           : 0.8663
  ROC-AUC      : 0.9809
  PR-AUC       : 0.8731
  Threshold    : 0.824807
  True frauds  : 98
  Pred frauds  : 89

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.91      0.83      0.87        98

    accuracy                           1.00     56962
   macro avg       0.95      0.91      0.93     56962
weighted avg       1.00      1.00      1.00     56962


Top 15 Feature Importances:
  V14                 : 0.4174
  ae_mae              : 0.0646
  ae_max_err          : 0.0514
  V17_abs             : 0.0498
  ae_mse              : 0.0457
  V14_abs             : 0.0445
  V4                  : 0.0335
  AmountSq            : 0.0173
  V20                 : 0.0163
  V13                 : 0.0163
  Amountlog           : 0.0153
  V8                  : 0.0144
  V10                 : 0.0

In [9]:
# DEBUG — XGBoost+AE diagnostics
print('=== XGBOOST+AE HYBRID DIAGNOSTICS ===')
print(f'AE architecture: {ae_model.count_params()} parameters')
print(f'AE features contribution (feature importance):')
for fname in ae_feature_names:
    idx = all_feature_cols.index(fname)
    print(f'  {fname}: importance={importances[idx]:.4f}')
ae_total = sum(importances[all_feature_cols.index(f)] for f in ae_feature_names)
print(f'  AE features total importance: {ae_total:.4f} ({100*ae_total/importances.sum():.1f}%)')
print()
print(f'XGBoost best iteration: {xgb_model.best_iteration}')
print(f'Threshold: {best_threshold:.6f}')
print(f'Score distribution (test):')
for pct in [50, 90, 95, 99, 99.5, 99.9]:
    print(f'  P{pct}: {np.percentile(test_probs, pct):.6f}')
print()
# Fraud vs normal score separation
fraud_scores = test_probs[y_test == 1]
normal_scores = test_probs[y_test == 0]
print(f'Normal scores: mean={normal_scores.mean():.6f}, max={normal_scores.max():.6f}')
print(f'Fraud scores:  mean={fraud_scores.mean():.6f}, min={fraud_scores.min():.6f}')
print(f'Separation gap: {fraud_scores.min() - normal_scores.max():.6f}')
# Note on variance
print()
print('NOTE: F1 variance of ~0.01-0.02 between runs is EXPECTED due to')
print('stochastic AE weight initialization and dropout during training.')
print('The AE component uses neural networks which are inherently non-deterministic')
print('even with fixed random seeds across different TF versions/hardware.')


=== XGBOOST+AE HYBRID DIAGNOSTICS ===
AE architecture: 4368 parameters
AE features contribution (feature importance):
  ae_mse: importance=0.0457
  ae_mae: importance=0.0646
  ae_max_err: importance=0.0514
  AE features total importance: 0.1617 (16.2%)

XGBoost best iteration: 954
Threshold: 0.824807
Score distribution (test):
  P50: 0.000013
  P90: 0.000150
  P95: 0.000376
  P99: 0.003569
  P99.5: 0.011079
  P99.9: 0.999539

Normal scores: mean=0.000476, max=0.998586
Fraud scores:  mean=0.845647, min=0.000010
Separation gap: -0.998576

NOTE: F1 variance of ~0.01-0.02 between runs is EXPECTED due to
stochastic AE weight initialization and dropout during training.
The AE component uses neural networks which are inherently non-deterministic
even with fixed random seeds across different TF versions/hardware.


## Cell 8 — Coordinator-Compatible Export

In [10]:
# Cell 8 — Coordinator-Compatible Export

artifact = {
    'xgb_model': xgb_model,
    'ae_model_weights': ae_model.get_weights(),
    'ae_model_config': ae_model.get_config(),
    'ae_scaler': ae_scaler,
    'base_feature_cols': base_feature_cols,
    'ae_feature_names': ae_feature_names,
    'all_feature_cols': all_feature_cols,
    'threshold': best_threshold,
    'test_indices': idx_test,
    'dataset': 'creditcard',
    'modelname': 'xgb_ae_hybrid',
    'runid': RUNID,
    'notebooktag': NOTEBOOKTAG,
}
artifact_path = os.path.join(ARTIFACTDIR, 'xgb_ae_hybrid_creditcard.joblib')
joblib.dump(artifact, artifact_path)
print('Saved artifact:', artifact_path)

ae_model_path = os.path.join(ARTIFACTDIR, 'vanilla_ae.keras')
ae_model.save(ae_model_path)
print('Saved AE model:', ae_model_path)

exported = {}
summaryrows = []

cc_bundle = {
    'dataset': 'creditcard',
    'model': 'xgb_ae_hybrid',
    'protocol': 'strict point-wise holdout (20% test)',
    'entities': {
        'creditcard': {
            'entityid': 'creditcard',
            'scoresfull': np.asarray(test_probs, dtype=np.float32),
            'yfull': np.asarray(y_test, dtype=np.int8),
            'predfull': np.asarray(test_preds, dtype=np.int8),
            'rowid': np.arange(len(y_test), dtype=np.int64),
            'originalrowid': np.asarray(idx_test, dtype=np.int64),
            'threshold': best_threshold,
            'featurecolumns': all_feature_cols,
            'splitmeta': {
                'random_state': RANDOM_STATE,
                'test_size': 0.20,
                'val_size': 0.25,
                'total_rows': len(y_all),
                'train_rows': len(idx_train),
                'val_rows': len(idx_val),
                'test_rows': len(idx_test),
            },
        }
    }
}

cc_path = os.path.join(PREDICTIONSDIR, 'xgb_ae_hybrid_creditcard_strict.joblib')
joblib.dump(cc_bundle, cc_path)
exported['creditcard'] = cc_path
print('Saved prediction bundle:', cc_path)

summaryrows.append({
    'dataset': 'creditcard',
    'protocol': 'strict point-wise holdout (20% test)',
    'precision': precision_v, 'recall': recall_v, 'f1': f1_v,
    'rocauc': rocauc_v, 'prauc': prauc_v,
    'threshold': best_threshold,
    'n_anomalies_true': int(y_test.sum()),
    'n_anomalies_pred': int(test_preds.sum()),
})
summary = pd.DataFrame(summaryrows)
summary_path = os.path.join(PREDICTIONSDIR, 'xgb_ae_hybrid_summary.csv')
summary.to_csv(summary_path, index=False)
print('Saved summary:', summary_path)
display(summary)

manifest = {
    'runid': RUNID, 'driveroot': DRIVEROOT, 'notebooktag': NOTEBOOKTAG,
    'modelfamily': 'xgb_ae_hybrid', 'exportprotocol': 'ensembleexportv2',
    'artifactsdir': ARTIFACTDIR, 'predictionsdir': PREDICTIONSDIR,
    'exports': exported, 'summarycsv': summary_path,
    'creditcard_originalrowid_included': True,
    'datasets': ['creditcard'], 'smd_dropped': True,
    'ae_reconstruction_features': ae_feature_names,
    'components': ['vanilla_autoencoder', 'xgboost_classifier'],
}
manifest_path = os.path.join(PREDICTIONSDIR, 'xgb_ae_hybrid_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print('Saved manifest:', manifest_path)

print()
print('All exports saved to:', PREDICTIONSDIR)
print('Done.')


Saved artifact: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/xgb_ae_hybrid/artifacts/xgb_ae_hybrid_creditcard.joblib
Saved AE model: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/xgb_ae_hybrid/artifacts/vanilla_ae.keras
Saved prediction bundle: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/xgb_ae_hybrid/predictions/xgb_ae_hybrid_creditcard_strict.joblib
Saved summary: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/xgb_ae_hybrid/predictions/xgb_ae_hybrid_summary.csv


,dataset,protocol,precision,recall,f1,rocauc,prauc,threshold,n_anomalies_true,n_anomalies_pred
0,creditcard,strict point-wise holdout (20% test),0.910112,0.826531,0.86631,0.980868,0.873092,0.824807,98,89


Saved manifest: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/xgb_ae_hybrid/predictions/xgb_ae_hybrid_manifest.json

All exports saved to: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/xgb_ae_hybrid/predictions
Done.
